In [1]:
import polars as pl

In [2]:
inicial = (
    pl.scan_parquet(r"C:\Users\diogo.durao\Documents\202501_NovoBolsaFamilia.parquet")
    .rename({
        'M�S COMPET�NCIA': 'MES_COMPETENCIA',
        'M�S REFER�NCIA': 'MES_REFERENCIA',
        'C�DIGO MUNIC�PIO SIAFI': 'CODIGO_MUNICIPIO_SIAFI',
        'NOME MUNIC�PIO': 'NOME_MUNICIPIO',
    })
    .with_columns(
        pl.col("VALOR PARCELA")
        .str.replace(",", ".")   
        .cast(pl.Float64)          
    )
    .with_columns(
    pl.col("MES_REFERENCIA")
    .cast(pl.Utf8)  # Primeiro converter para string
    .str.strptime(pl.Date, "%Y%m")  # Formato YYYYMM
    .alias("DATA_REF")
)
)

display(inicial.collect())

MES_COMPETENCIA,MES_REFERENCIA,UF,CODIGO_MUNICIPIO_SIAFI,NOME_MUNICIPIO,CPF FAVORECIDO,NIS FAVORECIDO,NOME FAVORECIDO,VALOR PARCELA,DATA_REF
i64,i64,str,i64,str,str,i64,str,f64,date
202501,202308,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0,2023-08-01
202501,202309,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0,2023-09-01
202501,202310,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0,2023-10-01
202501,202311,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0,2023-11-01
202501,202312,"""SP""",7071,"""SANTOS""","""***.085.106-**""",20643890445,"""FERNANDA RAMOS TEIXEIRA""",650.0,2023-12-01
…,…,…,…,…,…,…,…,…,…
202501,202501,"""TO""",9643,"""XAMBIOA""","""""",16640890691,"""ZEIA DE SOUZA LUCIO""",750.0,2025-01-01
202501,202501,"""TO""",9643,"""XAMBIOA""","""***.273.191-**""",20644881997,"""ZENILDE ALVES DOS SANTOS""",600.0,2025-01-01
202501,202501,"""TO""",9643,"""XAMBIOA""","""""",19058661973,"""ZENOLIA RAMOS DA SILVA CARVALH…",600.0,2025-01-01


In [3]:
# 1. Calcular os limites (Q1 e Q3) baseados no Valor Médio por Município
# (Assumindo que 'df_municipios' já existe da etapa anterior. Se não, recrie-o agrupando o LazyFrame)
df_municipios = inicial.group_by("NOME_MUNICIPIO", "UF").agg(pl.col("VALOR PARCELA").mean().alias("Valor_Medio_Pago"))

# Precisamos coletar para calcular os quartis exatos da distribuição das cidades
df_agregado = df_municipios.collect()

q1 = df_agregado.select(pl.col("Valor_Medio_Pago").quantile(0.25)).item()
q3 = df_agregado.select(pl.col("Valor_Medio_Pago").quantile(0.75)).item()

print(f"Municípios abaixo de R$ {q1:.2f} são 'Baixo Repasse'")
print(f"Municípios acima de R$ {q3:.2f} são 'Alto Repasse'")

# 2. Criar a coluna "Etiqueta" (Classificação) e Filtrar apenas os extremos
# Conforme o roteiro: "Concatenamos os resultados adicionando uma coluna nova 'etiqueta' para indicar se o município é 'Alto Repasse' ou 'Baixo Repasse'."
df_export = df_agregado.with_columns(
    pl.when(pl.col("Valor_Medio_Pago") > q3).then(pl.lit("Alto Repasse"))
    .when(pl.col("Valor_Medio_Pago") < q1).then(pl.lit("Baixo Repasse"))
    .otherwise(pl.lit("Médio")) # Apenas para preencher, mas vamos filtrar
    .alias("Etiqueta_Classificacao")
).filter(
    (pl.col("Etiqueta_Classificacao") == "Alto Repasse") | 
    (pl.col("Etiqueta_Classificacao") == "Baixo Repasse")
)

# 3. Exportar para CSV (Simulando a entrega para o time de Dataviz)
df_export.write_csv("bolsa_familia_extremos_municipios.csv")

print("Arquivo 'bolsa_familia_extremos_municipios.csv' gerado com sucesso!")
print(df_export.head())

Municípios abaixo de R$ 650.96 são 'Baixo Repasse'
Municípios acima de R$ 679.96 são 'Alto Repasse'
Arquivo 'bolsa_familia_extremos_municipios.csv' gerado com sucesso!
shape: (5, 4)
┌────────────────┬─────┬──────────────────┬────────────────────────┐
│ NOME_MUNICIPIO ┆ UF  ┆ Valor_Medio_Pago ┆ Etiqueta_Classificacao │
│ ---            ┆ --- ┆ ---              ┆ ---                    │
│ str            ┆ str ┆ f64              ┆ str                    │
╞════════════════╪═════╪══════════════════╪════════════════════════╡
│ MEDICILANDIA   ┆ PA  ┆ 698.267096       ┆ Alto Repasse           │
│ BREJO          ┆ MA  ┆ 706.064507       ┆ Alto Repasse           │
│ TAQUARITUBA    ┆ SP  ┆ 628.593679       ┆ Baixo Repasse          │
│ EIRUNEPE       ┆ AM  ┆ 772.507497       ┆ Alto Repasse           │
│ SAPEACU        ┆ BA  ┆ 646.761628       ┆ Baixo Repasse          │
└────────────────┴─────┴──────────────────┴────────────────────────┘
